## Converte Modelo

In [1]:
import numpy as np
import tensorflow as tf
import keras
from keras import models
import pickle

In [ ]:
keras_model_path = 'modelos\model_1d_CNN_mov-imag.keras'
keras_model = keras.models.load_model(keras_model_path)

with open('Dados_EEG\dados_EEG1.pkl', mode='rb') as f:
  X_train, y_train, X_test, y_test, CANAIS, TAM_JANELA= pickle.load(f)


def representative_dataset_gen():
  for i in range(200):
    amostra = X_train[i]
    amostra_batch = np.expand_dims(amostra, axis=0)   # (640, 5) vira (1, 640, 5)
    yield [amostra_batch.astype(np.float32)]  #uma amostra por loop

In [14]:
converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8  
converter.inference_output_type = tf.int8

In [15]:
tflite_model = converter.convert()

tflite_model_path = 'modelos/model_mov-imag.tflite'
with open(tflite_model_path, 'wb') as f:
  f.write(tflite_model)
print(f"Modelo salvo em: {tflite_model_path}")

INFO:tensorflow:Assets written to: C:\Users\eduar\AppData\Local\Temp\tmpzxf9h2ad\assets


INFO:tensorflow:Assets written to: C:\Users\eduar\AppData\Local\Temp\tmpzxf9h2ad\assets


Saved artifact at 'C:\Users\eduar\AppData\Local\Temp\tmpzxf9h2ad'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 640, 5), dtype=tf.float32, name='input_layer_38')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  1193883180432: TensorSpec(shape=(1, 1, 5), dtype=tf.float32, name=None)
  1193883181968: TensorSpec(shape=(1, 1, 5), dtype=tf.float32, name=None)
  1193883174288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1193883175248: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1193883173136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1193883176400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1193883176208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1193883175632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1193883176784: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1193883176592: TensorSpec(shape=(), dtype=tf.resource, name=None

C:\Users\eduar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tensorflow\lite\python\convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Modelo salvo em: modelos/model_mov-imag.tflite


Verifica conversão

In [ ]:
import tensorflow as tf
import numpy as np

def testar_modelo_eeg_tflite(caminho_modelo_tflite, classe_teste, X_test, y_test):
    """
    Carrega um modelo TFLite de EEG, executa a inferência em uma amostra específica
    do dataset de teste (fornecido) e exibe o resultado e o sinal.
    
    Args:
        caminho_modelo_tflite (str): Caminho para o arquivo .tflite.
        classe_teste (int): A classe que se deseja testar (0, 1 ou 2).
        X_test (np.ndarray): O array de dados de teste.
        y_test (np.ndarray): O array de rótulos de teste.
    """

    indices_da_classe = np.where(y_test == classe_teste)[0]
    if len(indices_da_classe) == 0:
        print(f"Erro: Não foi possível encontrar uma amostra da classe {classe_teste} no dataset de teste.")
        return

    # amostra aleatória dentro da classe 
    indice_aleatorio = np.random.choice(indices_da_classe)
    sinal_teste = X_test[indice_aleatorio]
    label_real = y_test[indice_aleatorio]
 
    # --- Carregar Interpretador TFLite ---
    interpreter = tf.lite.Interpreter(model_path=caminho_modelo_tflite)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    # --- Preparar o Sinal ---
    sinal_para_inferencia = np.expand_dims(sinal_teste, axis=0)

    if input_details['dtype'] == np.int8:
        input_scale, input_zero_point = input_details['quantization']
        sinal_para_inferencia = (sinal_para_inferencia / input_scale) + input_zero_point
        sinal_para_inferencia = sinal_para_inferencia.astype(np.int8)
    else: # Garante que é float32 se não for quantizado
        sinal_para_inferencia = sinal_para_inferencia.astype(np.float32)


    # --- Inferência ---
    interpreter.set_tensor(input_details['index'], sinal_para_inferencia)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details['index'])
    classe_prevista = np.argmax(output_data)

    # --- Resultados ---
    print(f"--- TESTANDO A CLASSE: {classe_teste} ---")
    print(f"Modelo: '{caminho_modelo_tflite}'")
    print(f"Saída Bruta do Modelo (Logits/Quantizados): {output_data[0]}")
    print(f"Resultado da Previsão: Classe {classe_prevista}")
    print(f"Resultado Real:       Classe {label_real}")
    print("---------------------------------")

testar_modelo_eeg_tflite(tflite_model_path, classe_teste=2, X_test=X_test, y_test=y_test)


NameError: name 'tflite_model_path' is not defined

Convertendo para C++

In [40]:
import os

def convert_binary_to_c_array(tflite_path, array_name = "model_tflite"):
    """
    Lê um arquivo binário e o converte para um source e um header C/C++.

    Args:
        tflite_path (str): O caminho para o arquivo binário de entrada.
    """
    try:
        with open(tflite_path, 'rb') as f:
            content = f.read()

    except FileNotFoundError:
        print(f'Erro: Arquivo "{tflite_path}" não encontrado.')
        return

    
    base_name = os.path.splitext(tflite_path)[0]
    cc_path = base_name + '.cc'
    h_path = base_name + '.h'
    file_size = len(content)

    # --- Geração Header (.h) ---
    header_guard = f'{base_name.upper()}_H_'
    
    with open(h_path, 'w') as f:
        f.write(f'#ifndef {header_guard}\n')
        f.write(f'#define {header_guard}\n\n')
        f.write(f'extern const unsigned char {array_name}[];\n')
        f.write(f'extern const unsigned int {array_name}_len;\n\n')
        f.write(f'#endif // {header_guard}\n')

    print(f'Arquivo de header "{h_path}" gerado com sucesso!')

    # --- Geração Source (.cc) ---
    with open(cc_path, 'w') as f:
        f.write(f'#include "{h_path}"\n\n')
        f.write(f'alignas(16) const unsigned char {array_name}[] = {{\n')

        hex_values = [f'0x{byte:02x}' for byte in content]

        for i in range(0, len(hex_values), 12):
            line = ', '.join(hex_values[i:i+12])
            f.write(f'  {line},\n')

        f.write(f'}};\n\n')
        f.write(f'const unsigned int {array_name}_len = {file_size};\n')

    print(f'Arquivo de source "{cc_path}" gerado com sucesso!')
    print(f'\n--- Resumo ---')
    print(f'Nome da variável do array: {array_name}')
    print(f'Nome da variável de tamanho: {array_name}_len')
    print(f'Para usar em seu código C/C++, inclua o header: #include "{os.path.basename(h_path)}"')

convert_binary_to_c_array(tflite_model_path)

Arquivo de header "modelos/model_mov-imag.h" gerado com sucesso!
Arquivo de source "modelos/model_mov-imag.cc" gerado com sucesso!

--- Resumo ---
Nome da variável do array: model_tflite
Nome da variável de tamanho: model_tflite_len
Para usar em seu código C/C++, inclua o header: #include "model_mov-imag.h"


In [7]:
import tensorflow as tf
import numpy as np

interpreter = tf.lite.Interpreter(model_path="modelos\model_mov-imag.tflite")
interpreter.allocate_tensors()
# --- Operadores ---
valid_ops = set() #tira duplicados
ignor_ops = {"DELEGATE", "CUSTOM"} # Ignorar

for op in interpreter._get_ops_details():
    op_name = op['op_name']
    if op_name not in ignor_ops:
        valid_ops.add(op_name)

# --- Tensor Arena ---
tensor_details = interpreter.get_tensor_details()
total_tensor_size = 0
for tensor in tensor_details:
    shape = tensor['shape']
    num_elements = np.prod(shape)
    dtype_size = np.dtype(tensor['dtype']).itemsize
    total_tensor_size += num_elements * dtype_size


print("Operadores que precisam ser registrados no TFLite Micro:")
print("---------------")
for op_name in sorted(valid_ops):
    print(op_name)
print("---------------\n")

print(f"Tamanho estimado da Tensor Arena:")
print(f"{total_tensor_size / 1024:.2f} KB\n")  

Operadores que precisam ser registrados no TFLite Micro:
---------------
ADD
CONV_2D
EXPAND_DIMS
FULLY_CONNECTED
MAX_POOL_2D
MUL
RESHAPE
SUB
---------------

Tamanho estimado da Tensor Arena:
292.42 KB



In [ ]:
# visualuzar parametros
print("\n--- Parâmetros de Quantização Gerados ---")
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()


input_details = interpreter.get_input_details()[0]


scale, zero_point = input_details['quantization']

print(f"Input Scale (escala): {scale}")
print(f"Input Zero Point (ponto_zero): {zero_point}")
print("USE ESTES VALORES NO SEU SCRIPT DE ENVIO!")
print("-------------------------------------------\n")


--- Parâmetros de Quantização Gerados ---
Input Scale (escala): 3.9294116497039795
Input Zero Point (ponto_zero): 9
USE ESTES VALORES NO SEU SCRIPT DE ENVIO!
-------------------------------------------



C:\Users\eduar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
